# Flood-warning indicators in South Sudan
## Historical patterns, recent-year testing, and lead-time evaluation

**Research question:** To what extent do rainfall and runoff indicators from local and provisional upstream zones help identify unusual flood detections in Ayod, Bor South, and Malakal, and how does performance hold up on recent unseen years?

This notebook is designed to be reproducible and deliberately conservative:
- It loads one year at a time to limit memory use.
- It does not treat missing flood-mask dates as flood-free.
- It keeps 2023–2025 out of model development.
- It labels outputs as exploratory because HydroBASINS candidates are not yet hydrologically validated.
- A detected flood-mask observation is a proxy outcome, not verified flood onset or a real-time operational warning.

**Important:** Before interpreting upstream results, verify that the provisional HydroBASINS polygons represent meaningful contributing areas for each research area. Their IDs below are candidate zones from earlier exploration, not confirmed catchments.

In [1]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
Using cached scikit_learn-1.9.1-cp312-cp312-win_amd64.whl (8.3 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ----------

In [2]:
from pathlib import Path
import os
import importlib.util
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    balanced_accuracy_score, average_precision_score
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "raw_data").exists():
    PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

RAW_DIR = PROJECT_ROOT / "raw_data"
PROCESSED_DIR = PROJECT_ROOT / "processing_data"
HYDRO_DIR = RAW_DIR / "hydrology"
RAIN_DIR = RAW_DIR / "rainfall and runoff"
FLOOD_DIR = RAW_DIR / "flood_masks"
ADMIN_DIR = RAW_DIR / "Administrative boundaries"
OUTPUT_DIR = PROCESSED_DIR / "upstream_rainfall_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_AREAS = ["Ayod", "Bor South", "Malakal"]
TRAIN_YEARS = list(range(2000, 2020))
VALIDATION_YEARS = list(range(2020, 2023))
TEST_YEARS = [2023, 2024, 2025]
ALL_YEARS = TRAIN_YEARS + VALIDATION_YEARS + TEST_YEARS
ROLLING_WINDOWS = [1, 3, 7, 14]
WARNING_HORIZON_DAYS = 3
PROBABILITY_THRESHOLD = 0.5

print("Project root:", PROJECT_ROOT)
print("ERA5 files:", len(list(RAIN_DIR.glob("ERA5_*.nc"))))
print("Output directory:", OUTPUT_DIR)

Project root: c:\Users\20242408\OneDrive - TU Eindhoven\Documenten\DATA SCIENCE BACHELOR\Jaar 3\Q1\Capstone data challenge\JBG060_ZHL_2026
ERA5 files: 26
Output directory: c:\Users\20242408\OneDrive - TU Eindhoven\Documenten\DATA SCIENCE BACHELOR\Jaar 3\Q1\Capstone data challenge\JBG060_ZHL_2026\processing_data\upstream_rainfall_outputs


In [3]:
# Load the project's existing data loader.
LOADING_FILE = PROCESSED_DIR / "loading.py"
if not LOADING_FILE.exists():
    raise FileNotFoundError(f"Expected project loader at {LOADING_FILE}")

loading_spec = importlib.util.spec_from_file_location("project_loading", LOADING_FILE)
project_loading = importlib.util.module_from_spec(loading_spec)
loading_spec.loader.exec_module(project_loading)

# Load administrative boundaries and keep the three study areas.
admin2_path = ADMIN_DIR / "ssd_admin2.geojson"
admin2 = gpd.read_file(admin2_path)
study_areas = admin2.loc[admin2["adm2_name"].isin(STUDY_AREAS)].copy()
study_areas = study_areas.set_index("adm2_name").loc[STUDY_AREAS].reset_index()

if len(study_areas) != len(STUDY_AREAS):
    raise ValueError("One or more requested study areas were not found in ssd_admin2.geojson")

print("Study-area CRS:", study_areas.crs)
display(study_areas[["adm2_name"]])

Study-area CRS: EPSG:4326


,adm2_name
0,Ayod
1,Bor South
2,Malakal


## 1. Audit source-data coverage

Check which years exist before choosing the final split. The proposed 2023–2025 test period should be revised if any year is missing or flood observations are too sparse.

In [4]:
era5_files = sorted(RAIN_DIR.glob("ERA5_*.nc"))
era5_years = sorted(int(p.stem.split("_")[-1]) for p in era5_files)
flood_subdirs = [FLOOD_DIR / "compact_recurring", FLOOD_DIR / "compact_unusual"]

flood_files_by_year = {}
for year in ALL_YEARS:
    year_files = []
    for folder in flood_subdirs:
        if folder.exists():
            year_files.extend([p for p in folder.glob("*.parquet") if str(year) in p.name])
    flood_files_by_year[year] = len(year_files)

coverage_audit = pd.DataFrame({
    "year": ALL_YEARS,
    "era5_available": [year in era5_years for year in ALL_YEARS],
    "flood_files_found_by_filename": [flood_files_by_year[y] for y in ALL_YEARS],
    "period": [
        "train" if y in TRAIN_YEARS else "validation" if y in VALIDATION_YEARS else "test"
        for y in ALL_YEARS
    ]
})
display(coverage_audit)
print("Missing ERA5 years:", sorted(set(ALL_YEARS) - set(era5_years)))
print("Note: parquet filename counts are a preliminary audit only; loader-level coverage is checked below.")

,year,era5_available,flood_files_found_by_filename,period
0,2000,True,4,train
1,2001,True,4,train
2,2002,True,4,train
3,2003,True,4,train
4,2004,True,4,train
5,2005,True,4,train
6,2006,True,4,train
7,2007,True,4,train
8,2008,True,4,train
9,2009,True,4,train


Missing ERA5 years: []
Note: parquet filename counts are a preliminary audit only; loader-level coverage is checked below.


## 2. Configure provisional upstream zones

These are the candidate HydroBASINS level-8 polygons shortlisted during exploration. They are **not confirmed hydrological catchments**. The notebook calculates upstream-zone rainfall as a sensitivity experiment and keeps local-area rainfall as a comparison.

If catchment validation is not completed, report the upstream results as provisional and do not describe them as verified contributing catchments.

In [5]:
HYDROBASINS_DIR = HYDRO_DIR / "hydrobasins_africa"
BASIN_LEVEL = 8
PROVISIONAL_BASIN_IDS = {
    "Ayod": [1080886390, 1080874940, 1080874930],
    "Bor South": [1080947990, 1080951620],
    "Malakal": [1080826450, 1080826310, 1080820600],
}

basin_path = HYDROBASINS_DIR / f"hybas_af_lev{BASIN_LEVEL:02d}_v1c.shp"
basins = gpd.read_file(basin_path)
basins["HYBAS_ID"] = basins["HYBAS_ID"].astype("int64")

zone_rows = []
for area, ids in PROVISIONAL_BASIN_IDS.items():
    for basin_id in ids:
        match = basins.loc[basins["HYBAS_ID"].eq(int(basin_id))]
        if not match.empty:
            zone_rows.append({"region": area, "zone_id": int(basin_id), "geometry": match.geometry.iloc[0]})

upstream_zones = gpd.GeoDataFrame(zone_rows, geometry="geometry", crs=basins.crs)
print("Found provisional basin polygons:", len(upstream_zones))
display(upstream_zones[["region", "zone_id"]])

Found provisional basin polygons: 8


,region,zone_id
0,Ayod,1080886390
1,Ayod,1080874940
2,Ayod,1080874930
3,Bor South,1080947990
4,Bor South,1080951620
5,Malakal,1080826450
6,Malakal,1080826310
7,Malakal,1080820600


## 3. Build grid-cell selections

ERA5 precipitation (`tp`) and runoff (`ro`) are loaded in metres per day and converted to millimetres per day. For local areas, use grid-cell centres inside the administrative polygon; if none exist, use the nearest cell. For provisional upstream zones, combine all listed candidate polygons for that region and select grid-cell centres inside the union.

A zone with no ERA5 grid centres is flagged rather than silently substituted.

In [6]:
# Load one year to obtain the grid coordinates; reuse this grid for all years.
first_year = next((y for y in ALL_YEARS if y in era5_years), None)
if first_year is None:
    raise FileNotFoundError("No ERA5 yearly files found for the configured analysis period.")

grid_sample = project_loading.load_rainfall_runoff(np.array([first_year]))
lon2d, lat2d = np.meshgrid(grid_sample.longitude.values, grid_sample.latitude.values)
grid_points = gpd.GeoDataFrame(
    {"latitude": lat2d.ravel(), "longitude": lon2d.ravel()},
    geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
    crs="EPSG:4326"
)

study_areas_wgs = study_areas.to_crs("EPSG:4326")
if upstream_zones.crs != "EPSG:4326":
    upstream_zones = upstream_zones.to_crs("EPSG:4326")

local_cell_indices = {}
for _, area in study_areas_wgs.iterrows():
    mask = grid_points.geometry.within(area.geometry)
    indices = np.flatnonzero(mask.to_numpy())
    if len(indices) == 0:
        # Nearest grid cell fallback, mainly relevant for very small polygons.
        distances = grid_points.to_crs("EPSG:32636").geometry.distance(
            gpd.GeoSeries([area.geometry], crs="EPSG:4326").to_crs("EPSG:32636").iloc[0]
        )
        indices = np.array([int(distances.idxmin())])
        print(f"WARNING: {area['adm2_name']} uses nearest ERA5 grid cell.")
    local_cell_indices[area["adm2_name"]] = indices

upstream_cell_indices = {}
for area in STUDY_AREAS:
    zone = upstream_zones.loc[upstream_zones["region"].eq(area)]
    if zone.empty:
        upstream_cell_indices[area] = np.array([], dtype=int)
        continue
    union_geom = zone.geometry.unary_union
    mask = grid_points.geometry.within(union_geom)
    upstream_cell_indices[area] = np.flatnonzero(mask.to_numpy())
    if len(upstream_cell_indices[area]) == 0:
        print(f"WARNING: No ERA5 grid centres in provisional upstream zone for {area}.")

print("Local grid-cell counts:", {k: len(v) for k, v in local_cell_indices.items()})
print("Provisional upstream grid-cell counts:", {k: len(v) for k, v in upstream_cell_indices.items()})

Daily total precipitation and runoff from ERA5 reanalysis loaded from dates 2000-01-01 to 2000-12-31
<xarray.Dataset> Size: 24MB
Dimensions:     (valid_time: 366, latitude: 145, longitude: 57)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2000-01-01 ... 2000-12-31
  * latitude    (latitude) float64 1kB 33.0 32.75 32.5 32.25 ... -2.5 -2.75 -3.0
  * longitude   (longitude) float64 456B 23.0 23.25 23.5 ... 36.5 36.75 37.0
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 12MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
    ro          (valid_time, latitude, longitude) float32 12MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:    

C:\Users\20242408\AppData\Local\Temp\ipykernel_12592\1565043055.py:37: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  union_geom = zone.geometry.unary_union


## 4. Aggregate daily rainfall/runoff for all available years

This cell loads ERA5 one year at a time and produces a long daily table. It avoids keeping all years of gridded data in memory.

In [7]:
rain_parts = []
missing_era5_years = []

for year in ALL_YEARS:
    if year not in era5_years:
        missing_era5_years.append(year)
        continue

    print(f"Loading ERA5 {year}...")
    ds = project_loading.load_rainfall_runoff(np.array([year]))
    for area in STUDY_AREAS:
        for zone_type, indices in [
            ("local", local_cell_indices[area]),
            ("upstream_provisional", upstream_cell_indices[area]),
        ]:
            if len(indices) == 0:
                continue
            flat_lat_idx = indices // len(ds.longitude)
            flat_lon_idx = indices % len(ds.longitude)
            selected = ds[["tp", "ro"]].isel(
                latitude=xr.DataArray(flat_lat_idx, dims="cell"),
                longitude=xr.DataArray(flat_lon_idx, dims="cell")
            ).mean(dim="cell") * 1000
            part = selected.to_dataframe().reset_index()
            part = part.rename(columns={"valid_time": "date", "tp": "rain_mm", "ro": "runoff_mm"})
            part["region"] = area
            part["zone_type"] = zone_type
            part["year"] = year
            rain_parts.append(part[["date", "region", "zone_type", "year", "rain_mm", "runoff_mm"]])
    ds.close()

rain_daily = pd.concat(rain_parts, ignore_index=True)
rain_daily["date"] = pd.to_datetime(rain_daily["date"])
rain_daily = rain_daily.drop_duplicates(["date", "region", "zone_type"]).sort_values(["region", "zone_type", "date"])
display(rain_daily.groupby(["region", "zone_type"])["date"].agg(["min", "max", "count"]))
display(rain_daily.groupby(["region", "zone_type"])[["rain_mm", "runoff_mm"]].agg(["mean", "median", "max"]).round(2))
print("Years without ERA5:", missing_era5_years)

Loading ERA5 2000...
Daily total precipitation and runoff from ERA5 reanalysis loaded from dates 2000-01-01 to 2000-12-31
<xarray.Dataset> Size: 24MB
Dimensions:     (valid_time: 366, latitude: 145, longitude: 57)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2000-01-01 ... 2000-12-31
  * latitude    (latitude) float64 1kB 33.0 32.75 32.5 32.25 ... -2.5 -2.75 -3.0
  * longitude   (longitude) float64 456B 23.0 23.25 23.5 ... 36.5 36.75 37.0
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 12MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
    ro          (valid_time, latitude, longitude) float32 12MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forec

min        max  count
region    zone_type                                        
Ayod      local                2000-01-01 2025-12-31   9497
          upstream_provisional 2000-01-01 2025-12-31   9497
Bor South local                2000-01-01 2025-12-31   9497
          upstream_provisional 2000-01-01 2025-12-31   9497
Malakal   local                2000-01-01 2025-12-31   9497
          upstream_provisional 2000-01-01 2025-12-31   9497

rain_mm                   runoff_mm         \
                                  mean median        max      mean median   
region    zone_type                                                         
Ayod      local                   2.30   0.34  52.980000      0.11    0.0   
          upstream_provisional    2.31   0.36  38.139999      0.09    0.0   
Bor South local                   2.41   0.49  65.199997      0.09    0.0   
          upstream_provisional    2.43   0.47  87.279999      0.09    0.0   
Malakal   local                   1.97   0.08  42.570000      0.08    0.0   
          upstream_provisional    2.21   0.07  59.939999      0.12    0.0   

                                           
                                      max  
region    zone_type                        
Ayod      local                 11.060000  
          upstream_provisional   8.440000  
Bor South local                 21.480000  
          upstream_provisional  35.040001  
Malakal   local                  7.380000  
          upstream_provisional  15.740000

Years without ERA5: []


## 5. Load flood masks year by year and construct observed labels

The target is **unusual flood-mask detection**, not independently verified flood onset. We count unusual observations within each administrative area on dates represented in the flood-mask data. Dates with no mask records are excluded, not labelled negative.

The flood-mask loader may return millions of rows per year, so this cell filters each year's data to the bounding box around the study areas before doing point-in-polygon checks.

In [8]:
# Use a slightly expanded bounding box to avoid discarding points on polygon edges.
bounds = study_areas_wgs.total_bounds
minx, miny, maxx, maxy = bounds
bbox = (minx - 0.5, miny - 0.5, maxx + 0.5, maxy + 0.5)

flood_daily_parts = []
flood_loader_audit = []

for year in ALL_YEARS:
    print(f"Loading flood masks {year}...")
    try:
        masks = project_loading.load_flood_masks(np.array([year]))
    except Exception as exc:
        flood_loader_audit.append({"year": year, "status": "load_error", "rows": 0, "error": str(exc)})
        continue

    if masks is None or masks.empty:
        flood_loader_audit.append({"year": year, "status": "empty", "rows": 0, "error": ""})
        continue

    masks["date"] = pd.to_datetime(masks["date"])
    masks = masks.loc[
        masks["lon"].between(bbox[0], bbox[2]) &
        masks["lat"].between(bbox[1], bbox[3])
    ].copy()

    if masks.empty:
        flood_loader_audit.append({"year": year, "status": "no_points_near_areas", "rows": 0, "error": ""})
        continue

    points = gpd.GeoDataFrame(
        masks, geometry=gpd.points_from_xy(masks["lon"], masks["lat"]), crs="EPSG:4326"
    )

    for _, area in study_areas_wgs.iterrows():
        region = area["adm2_name"]
        inside = points.geometry.within(area.geometry)
        local = points.loc[inside]
        if local.empty:
            continue
        daily = local.groupby(["date", "flood_type"]).size().unstack(fill_value=0)
        daily = daily.rename(columns={0: "recurring_pixels", 1: "unusual_pixels"})
        for col in ["recurring_pixels", "unusual_pixels"]:
            if col not in daily:
                daily[col] = 0
        daily = daily[["recurring_pixels", "unusual_pixels"]].reset_index()
        daily["region"] = region
        daily["year"] = year
        daily["has_mask_observation"] = True
        flood_daily_parts.append(daily)

    flood_loader_audit.append({"year": year, "status": "loaded", "rows": len(masks), "error": ""})

flood_daily = pd.concat(flood_daily_parts, ignore_index=True) if flood_daily_parts else pd.DataFrame()
flood_audit = pd.DataFrame(flood_loader_audit)
display(flood_audit)
if flood_daily.empty:
    raise ValueError("No local flood-mask observations were loaded; inspect loader paths and year coverage.")

flood_daily["unusual_detected"] = flood_daily["unusual_pixels"].gt(0).astype(int)
flood_daily["total_detected_pixels"] = flood_daily["recurring_pixels"] + flood_daily["unusual_pixels"]
display(
    flood_daily.groupby(["region", "year"]).agg(
        observed_dates=("date", "nunique"),
        unusual_detection_dates=("unusual_detected", "sum"),
        unusual_pixels=("unusual_pixels", "sum")
    ).reset_index()
)

Loading flood masks 2000...
Loading flood masks 2001...
Loading flood masks 2002...
Loading flood masks 2003...
Loading flood masks 2004...
Loading flood masks 2005...
Loading flood masks 2006...
Loading flood masks 2007...
Loading flood masks 2008...
Loading flood masks 2009...
Loading flood masks 2010...
Loading flood masks 2011...
Loading flood masks 2012...
Loading flood masks 2013...
Loading flood masks 2014...
Loading flood masks 2015...
Loading flood masks 2016...
Loading flood masks 2017...
Loading flood masks 2018...
Loading flood masks 2019...
Loading flood masks 2020...
Loading flood masks 2021...
Loading flood masks 2022...
Loading flood masks 2023...
Loading flood masks 2024...
Loading flood masks 2025...


,year,status,rows,error
0,2000,loaded,5486,
1,2001,loaded,5286,
2,2002,loaded,43684,
3,2003,loaded,116090,
4,2004,loaded,98257,
5,2005,loaded,139790,
6,2006,loaded,468761,
7,2007,loaded,626204,
8,2008,loaded,391327,
9,2009,loaded,631637,


,region,year,observed_dates,unusual_detection_dates,unusual_pixels
0,Ayod,2000,67,53,345
1,Ayod,2001,69,64,754
2,Ayod,2002,152,134,5268
3,Ayod,2003,259,235,13358
4,Ayod,2004,282,251,8985
...,...,...,...,...,...
71,Malakal,2021,108,108,18427
72,Malakal,2022,144,144,20817
73,Malakal,2023,174,173,18572
74,Malakal,2024,98,98,1904


## 6. Build predictor features and future detection labels

Features use only data from dates before the prediction date. The label is whether at least one unusual detection occurs in the next 3 days, using only dates for which local mask records exist. To avoid calling unknown dates negatives, a training row is retained only when all dates in its future window have mask observations.

This conservative label rule may sharply reduce the sample. If so, report that limitation rather than weakening the missing-data rule without justification.

In [11]:
# Prepare the daily label calendar from dates with local mask records.
label_calendar = flood_daily[["region", "date", "unusual_detected", "has_mask_observation"]].copy()
label_calendar = label_calendar.drop_duplicates(["region", "date"]).sort_values(["region", "date"])

feature_parts = []
for (region, zone_type), group in rain_daily.groupby(["region", "zone_type"]):
    group = group.sort_values("date").copy()
    # Build lagged cumulative indicators using the prior day through prior N days.
    for window in ROLLING_WINDOWS:
        group[f"rain_prev_{window}d_mm"] = group["rain_mm"].shift(1).rolling(window, min_periods=window).sum()
        group[f"runoff_prev_{window}d_mm"] = group["runoff_mm"].shift(1).rolling(window, min_periods=window).sum()
    group["rain_change_7d_vs_prev7d"] = (
        group["rain_mm"].shift(1).rolling(7, min_periods=7).sum()
        - group["rain_mm"].shift(8).rolling(7, min_periods=7).sum()
    )
    group["month"] = group["date"].dt.month
    group["year"] = group["date"].dt.year
    group["region"] = region
    group["zone_type"] = zone_type
    feature_parts.append(group)

features = pd.concat(feature_parts, ignore_index=True)
base_feature_cols = [f"{kind}_prev_{w}d_mm" for w in ROLLING_WINDOWS for kind in ["rain", "runoff"]] + ["rain_change_7d_vs_prev7d"]
local_features = features.loc[features["zone_type"].eq("local")].copy()
local_features = local_features[["date", "region", "year", "month", "rain_mm", "runoff_mm"] + base_feature_cols]
local_features = local_features.rename(columns={
    col: f"local_{col}" for col in ["rain_mm", "runoff_mm"] + base_feature_cols
})

upstream_features = features.loc[features["zone_type"].eq("upstream_provisional")].copy()
upstream_keep = ["date", "region", "rain_mm", "runoff_mm"] + base_feature_cols
upstream_features = upstream_features[upstream_keep].rename(columns={
    col: f"upstream_{col}" for col in ["rain_mm", "runoff_mm"] + base_feature_cols
})
local_features = local_features.merge(upstream_features, on=["date", "region"], how="left")

feature_cols = (
    [f"local_{col}" for col in ["rain_mm", "runoff_mm"] + base_feature_cols]
    + [f"upstream_{col}" for col in ["rain_mm", "runoff_mm"] + base_feature_cols]
    + ["month"]
)

# Construct a forward-looking label only when every calendar day in the horizon is observed.
labels_by_region = {}
for region, group in label_calendar.groupby("region"):
    group = group.sort_values("date").drop_duplicates("date").set_index("date")
    calendar = pd.date_range(group.index.min(), group.index.max(), freq="D")
    daily = group.reindex(calendar)
    daily.index.name = "date"
    observed = daily["has_mask_observation"].fillna(False).astype(bool)
    future_any = daily["unusual_detected"].shift(-1).rolling(WARNING_HORIZON_DAYS, min_periods=WARNING_HORIZON_DAYS).max().shift(-(WARNING_HORIZON_DAYS - 1))
    # Future window: t+1 ... t+horizon. Keep only fully observed windows.
    future_observed = pd.concat(
        [observed.shift(-step) for step in range(1, WARNING_HORIZON_DAYS + 1)], axis=1
    ).all(axis=1)
    label_frame = pd.DataFrame({
        "date": calendar,
        "region": region,
        "target_flood_next_horizon": future_any.to_numpy(),
        "future_window_observed": future_observed.to_numpy()
    })
    labels_by_region[region] = label_frame

labels = pd.concat(labels_by_region.values(), ignore_index=True)
model_data = local_features.merge(labels, on=["region", "date"], how="inner")
model_data = model_data.loc[
    model_data["future_window_observed"]
    & model_data["target_flood_next_horizon"].notna()
].copy()

model_data["target_flood_next_horizon"] = (
    model_data["target_flood_next_horizon"].astype(int)
)
model_data["year"] = model_data["date"].dt.year

display(model_data.groupby(["region", "year"])["target_flood_next_horizon"].agg(["count", "sum", "mean"]).reset_index())
print("Model rows:", len(model_data))

,region,year,count,sum,mean
0,Ayod,2000,20,19,0.950000
1,Ayod,2001,21,21,1.000000
2,Ayod,2002,89,89,1.000000
3,Ayod,2003,172,171,0.994186
4,Ayod,2004,208,207,0.995192
...,...,...,...,...,...
62,Malakal,2021,72,72,1.000000
63,Malakal,2022,117,117,1.000000
64,Malakal,2023,115,115,1.000000
65,Malakal,2024,60,60,1.000000


Model rows: 8401


## 7. Train, validate, and evaluate on recent years

The initial model is logistic regression with median imputation and standardization. For each region:
- fit on 2000–2019;
- choose the probability threshold on 2020–2022 (maximize F1, with a fallback of 0.5 if validation is unsuitable);
- evaluate once on 2023–2025.

The held-out years are not used to choose features or thresholds. If a split has too few positive/negative cases, the metrics are still reported with a warning and must not be overinterpreted.

In [12]:
def confusion_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan,
        "positive_cases": int(np.sum(y_true)),
        "n": int(len(y_true)),
    }

model_results = []
prediction_parts = []
model_objects = {}

for region in STUDY_AREAS:
    data = model_data.loc[model_data["region"].eq(region)].sort_values("date").copy()
    train = data[data["year"].isin(TRAIN_YEARS)].dropna(subset=["target_flood_next_horizon"])
    val = data[data["year"].isin(VALIDATION_YEARS)].dropna(subset=["target_flood_next_horizon"])
    test = data[data["year"].isin(TEST_YEARS)].dropna(subset=["target_flood_next_horizon"])

    print(f"{region}: train={len(train)}, validation={len(val)}, test={len(test)}")
    if len(train) < 50 or train["target_flood_next_horizon"].nunique() < 2:
        print(f"WARNING: Insufficient training examples/classes for {region}; skipping model.")
        continue
    if test.empty or test["target_flood_next_horizon"].nunique() < 2:
        print(f"WARNING: Test set for {region} is empty or contains one class; metrics will be limited.")

    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
    )
    model.fit(train[feature_cols], train["target_flood_next_horizon"])

    threshold = PROBABILITY_THRESHOLD
    if len(val) >= 20 and val["target_flood_next_horizon"].nunique() == 2:
        val_prob = model.predict_proba(val[feature_cols])[:, 1]
        thresholds = np.arange(0.10, 0.91, 0.05)
        threshold_scores = [
            f1_score(val["target_flood_next_horizon"], val_prob >= t, zero_division=0)
            for t in thresholds
        ]
        threshold = float(thresholds[int(np.argmax(threshold_scores))])

    if not test.empty:
        test_prob = model.predict_proba(test[feature_cols])[:, 1]
        test_pred = (test_prob >= threshold).astype(int)
        metrics = confusion_metrics(test["target_flood_next_horizon"].to_numpy(), test_pred)
        metrics.update({"region": region, "threshold": threshold, "test_years": ",".join(map(str, TEST_YEARS))})
        metrics["average_precision"] = (
            average_precision_score(test["target_flood_next_horizon"], test_prob)
            if test["target_flood_next_horizon"].nunique() == 2 else np.nan
        )
        model_results.append(metrics)
        pred = test[["date", "region", "target_flood_next_horizon"]].copy()
        pred["probability"] = test_prob
        pred["predicted_warning"] = test_pred
        pred["threshold"] = threshold
        prediction_parts.append(pred)
    model_objects[region] = model

metrics_df = pd.DataFrame(model_results)
predictions = pd.concat(prediction_parts, ignore_index=True) if prediction_parts else pd.DataFrame()
display(metrics_df.round(3))

Ayod: train=3208, validation=675, test=849
Bor South: train=2146, validation=525, test=598
Malakal: train=27, validation=191, test=182


,TN,FP,FN,TP,precision,recall,f1,balanced_accuracy,positive_cases,n,region,threshold,test_years,average_precision
0,0,0,303,546,1.0,0.643,0.783,NaN,849,849,Ayod,0.5,"2023,2024,2025",NaN
1,0,0,204,394,1.0,0.659,0.794,NaN,598,598,Bor South,0.5,"2023,2024,2025",NaN


In [15]:
[name for name in globals() if any(
    word in name.lower()
    for word in ["train", "test", "valid", "split", "model_data"]
)]

['TRAIN_YEARS',
 'VALIDATION_YEARS',
 'TEST_YEARS',
 'model_data',
 'train',
 'test',
 'test_prob',
 'test_pred']

In [16]:
print("TRAIN_YEARS:", TRAIN_YEARS)
print("VALIDATION_YEARS:", VALIDATION_YEARS)
print("TEST_YEARS:", TEST_YEARS)

for name, df in [("train", train), ("test", test)]:
    print(f"\n{name.upper()} — overall target distribution")
    print(df["target_flood_next_horizon"].value_counts(dropna=False).sort_index())

    print(f"\n{name.upper()} — by region and year")
    print(
        df.groupby(
            ["region", df["date"].dt.year],
            dropna=False
        )["target_flood_next_horizon"]
        .agg(["count", "sum", "mean", "nunique"])
    )

TRAIN_YEARS: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
VALIDATION_YEARS: [2020, 2021, 2022]
TEST_YEARS: [2023, 2024, 2025]

TRAIN — overall target distribution
target_flood_next_horizon
0     1
1    26
Name: count, dtype: int64

TRAIN — by region and year
              count  sum  mean  nunique
region  date                           
Malakal 2005      4    4   1.0        1
        2007      2    2   1.0        1
        2008      8    8   1.0        1
        2010      1    0   0.0        1
        2012      3    3   1.0        1
        2014      3    3   1.0        1
        2016      3    3   1.0        1
        2017      1    1   1.0        1
        2018      2    2   1.0        1

TEST — overall target distribution
target_flood_next_horizon
1    182
Name: count, dtype: int64

TEST — by region and year
              count  sum  mean  nunique
region  date                           
Malakal 2023    115  

In [17]:
model_data.groupby(
    ["region", model_data["date"].dt.year.isin(TRAIN_YEARS).map({
        True: "train",
        False: "later"
    })]
)["target_flood_next_horizon"].agg(
    rows="count",
    positives="sum",
    positive_rate="mean",
    classes="nunique"
)

rows  positives  positive_rate  classes
region    date                                          
Ayod      later  1524       1524       1.000000        1
          train  3208       3115       0.971010        2
Bor South later  1123       1123       1.000000        1
          train  2146       2066       0.962721        2
Malakal   later   373        373       1.000000        1
          train    27         26       0.962963        2

In [13]:
# Check the target distribution across each split and year.
for region, splits in split_data.items():
    print(f"\n{region}")
    for split_name, split_df in splits.items():
        print(f"\n{split_name}:")
        print(
            split_df.groupby(split_df["date"].dt.year)[
                "target_flood_next_horizon"
            ]
            .agg(["count", "sum", "mean", "nunique"])
        )

NameError: name 'split_data' is not defined

## 8. Estimate lead time and inspect model behaviour

The classification target asks whether an unusual detection occurs within the next three days. This means the target horizon is not the same as a measured lead time. The table below reports the fraction of test predictions that were warnings and the lead time is bounded by the chosen horizon; event-level lead-time analysis should be done only after flood detections have been grouped into distinct events.

Do not report the mean of all positive prediction days as independent warning events: consecutive warning days can refer to the same flood episode.

In [ ]:
if not predictions.empty:
    predictions["year"] = predictions["date"].dt.year
    display(
        predictions.groupby("region").agg(
            test_dates=("date", "count"),
            warnings_issued=("predicted_warning", "sum"),
            observed_future_flood_windows=("target_flood_next_horizon", "sum"),
            mean_predicted_probability=("probability", "mean")
        ).reset_index().round(3)
    )

    fig, axes = plt.subplots(len(predictions["region"].unique()), 1, figsize=(14, 3.5 * len(predictions["region"].unique())), sharex=True)
    if len(predictions["region"].unique()) == 1:
        axes = [axes]
    for ax, (region, group) in zip(axes, predictions.groupby("region")):
        ax.plot(group["date"], group["probability"], label="Predicted probability")
        ax.axhline(group["threshold"].iloc[0], linestyle="--", label="Threshold")
        ax.set_title(region)
        ax.set_ylabel("Probability")
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No predictions available. Check sample sizes and target coverage above.")

## 9. Save dashboard-ready outputs

Exports are marked exploratory and include the model split and target definition. Do not use the provisional upstream polygons as confirmed catchments in the final presentation until hydrological validation is complete.

In [ ]:
coverage_audit.to_csv(OUTPUT_DIR / "year_coverage_audit.csv", index=False)
flood_audit.to_csv(OUTPUT_DIR / "flood_loader_audit.csv", index=False)
rain_daily.to_csv(OUTPUT_DIR / "daily_rainfall_runoff_features.csv", index=False)
flood_daily.to_csv(OUTPUT_DIR / "daily_local_flood_mask_observations.csv", index=False)
model_data.to_csv(OUTPUT_DIR / "model_dataset_exploratory.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "heldout_test_metrics_exploratory.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "heldout_test_predictions_exploratory.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)
print("Important caveats:")
print("- Flood-mask detection is a proxy, not verified flood onset.")
print("- Missing observation dates are not labelled flood-free.")
print("- Upstream polygons are provisional and require hydrological validation.")
print("- 2023–2025 metrics are only meaningful if coverage and class counts are adequate.")